# Beauty research workflow: EDA → 5 Whys → Good Questions

Combines the deterministic tools used by [eda_tools_agent.ipynb](eda_tools_agent.ipynb) with the [5-whys methodology](~/.codex/skills/5-whys/SKILL.md) and [good-question methodology](~/.codex/skills/good-question/SKILL.md).

**Goal:** use catalog and review evidence to design a falsifiable LLM/customer–product-fit study. The skills guide analysis and experimental design; they are not Python agents. No LLM, API key or network request is used here. Original upstream notebook is preserved.

Use the **Beauty EDA (Python 3.12)** kernel and Run All. Requires the outputs of `scripts/beauty_eda.py` and `scripts/add_beauty_reviews.py`. Outputs overwrite `output/beauty_research_workflow/`. Run time depends on local resources. Conclusions and pilot plans are provisional; this notebook does not execute LLM experiments.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if not (root / 'scripts/beauty_research_workflow.py').exists():
    root = root.parent
assert (root / 'scripts/beauty_research_workflow.py').exists(), 'Open from project root or notebooks/'
sys.path.insert(0, str(root / 'scripts'))
from beauty_research_workflow import (
    OUT, load_data, run_eda, build_evidence, build_whys,
    build_questions, write_report,
)
from IPython.display import display, Markdown, Image
import pandas as pd

## 1. Validate sources and scope

Load all catalog records and review metadata. Exact duplicate review records are excluded from review statistics; raw records remain available. Hash the actual inputs and record package versions. Parent-ID matches do not verify variant attributes. Listing snapshots have unknown effective dates.

In [ ]:
data = load_data()
display(pd.DataFrame([{'population': k, 'rows': len(v)} for k,v in data.items()]))

## 2. Execute the original notebook’s EDA tools

`describe_dataset`: full catalog/review numeric profiles. `visualize_missing` and Sweetviz: uniform 2,000-product sample, seed 42. `generate_correlation_funnel`: 5,000-product availability diagnostic, seed 42. The target is observed price availability; price itself is excluded from predictors. Correlations do not explain missingness. Direct tool invocation makes this stage reproducible without an LLM choosing tools.

In [ ]:
tool_status = run_eda(data)
display(pd.DataFrame(tool_status).T)
display(Image(filename=str(OUT / 'matrix_plot.png')))
display(Image(filename=str(OUT / 'availability_funnel.png')))

## 3. Build the evidence ledger

Every fact records its numerator, denominator, source and interpretation boundary. Review-record, reviewer and product denominators must remain separate. Candidate mixed-rating products are not validated preference conflicts.

In [ ]:
evidence = build_evidence(data)
display(pd.DataFrame(evidence))

## 4. Apply 5 Whys without inventing causes

Start from observable constraints. Label verified rule consequences, measurement limitations, rival hypotheses and unknowns. Stop when evidence ends. Each chain includes a countermeasure and reverse check; no chain establishes why shoppers bought or disliked a product.

In [ ]:
chains = build_whys(evidence)
display(Markdown((OUT / 'five_whys.md').read_text()))

## 5. Apply Good Question: assumptions, rivals, falsifiers and pilots

Generate six candidate questions and rank them using explicit 1–5 planning judgments. Originality stays unscored pending a literature audit. Develop two provisional cards with competing hypotheses, independent evaluation, falsifiers, two-week pilots and rejection risks. These are experimental designs, not results or conference-readiness claims.

In [ ]:
questions, cards = build_questions()
display(pd.DataFrame([{'id': q['id'], 'question': q['question'], **q['scores'],
                       'planning_total': q['planning_total'], 'decision': q['decision']}
                      for q in questions]))
display(Markdown((OUT / 'research_questions.md').read_text()))

## 6. Publish the combined local report

**Next evidence:** independent human labels first; prospective shopper needs and fit judgments next; merchant experiments before commercial uplift claims. Do not derive both a need and its supposedly independent fit label from the same target review. Group product families and exact duplicates, isolate development/test sets, and time-align history. The existing 200-review annotation seed is rating-balanced and not population-representative.

In [ ]:
report = write_report(evidence, chains, questions, cards)
assert len(evidence) == 9 and len(chains) == 3 and len(questions) == 6 and len(cards) == 2
assert all(q['originality'] is None for q in questions)
assert all(0 <= e['numerator'] <= e['denominator'] for e in evidence)
print(f'Report: {report}')
print('Workflow checks passed. LLM evaluations remain unrun.')